In [1]:
# ── Cell 1 — imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)


In [3]:
# ── Cell 2 — load week 1 outputs ─────────────────────────────────────
demand_df  = pd.read_csv(r'F:\projectss\personal\blinkit_project\data\processed\demand_table.csv')
pincode_df = pd.read_csv(r'F:\projectss\personal\blinkit_project\data\raw\5c2f62fe-5afa-4119-a499-fec9d604d5bd.csv')

print("demand_table shape :", demand_df.shape)
print("columns            :", demand_df.columns.tolist())
print(demand_df.head(3))

demand_table shape : (79213, 10)
columns            : ['product_id', 'product_name', 'pincode', 'area_name', 'latitude', 'longitude', 'order_dow', 'order_hour_of_day', 'demand', 'demand_log']
   product_id   product_name  pincode area_name  latitude  longitude  \
0        4605  Yellow Onions   380006     Paldi     23.02      72.57   
1        4605  Yellow Onions   380006     Paldi     23.02      72.57   
2        4605  Yellow Onions   380006     Paldi     23.02      72.57   

   order_dow  order_hour_of_day  demand  demand_log  
0          0                  0     100        4.62  
1          0                  1     120        4.80  
2          0                  2      50        3.93  


In [10]:
# ── Cell 3 — get unique product+pincode combos ────────────────────────
combos = demand_df[
    ['product_id','product_name','pincode',
     'area_name','latitude','longitude']
].drop_duplicates().reset_index(drop=True)

print(f"Unique product+pincode combos: {len(combos)}")
print(combos.head(5))

Unique product+pincode combos: 500
   product_id   product_name  pincode    area_name  latitude  longitude
0        4605  Yellow Onions   380006        Paldi     23.02      72.57
1        4605  Yellow Onions   380009  Navrangpura     23.05      72.55
2        4605  Yellow Onions   380013         Gota     23.05      72.56
3        4605  Yellow Onions   380015    Satellite     23.03      72.52
4        4605  Yellow Onions   380019    Vastrapur     23.10      72.58


In [11]:
# ── Cell 4 — build 26 weeks of hourly rows with weekly noise ──────────
# THIS IS THE KEY FIX
# each week gets ±20% random noise on demand
# so lag_168 (same hour last week) is genuinely different from today

NUM_WEEKS  = 26
START_DATE = '2024-01-01'

all_rows = []

for week in range(NUM_WEEKS):
    week_start    = pd.Timestamp(START_DATE) + pd.Timedelta(weeks=week)
    # BETTER — more realistic variation
    noise_factor = np.random.uniform(0.60, 1.40) # different per week

    for _, combo in combos.iterrows():
        base = demand_df[
            (demand_df['product_id'] == combo['product_id']) &
            (demand_df['pincode']    == combo['pincode'])
        ][['order_dow','order_hour_of_day','demand']].copy()

        if base.empty:
            continue

        # apply weekly noise
        base['demand'] = (base['demand'] * noise_factor).clip(lower=0).round(0)

        for _, row in base.iterrows():
            dt = week_start + pd.Timedelta(
                days=int(row['order_dow']),
                hours=int(row['order_hour_of_day'])
            )
            all_rows.append({
                'datetime'         : dt,
                'week_number'      : week + 1,
                'product_id'       : combo['product_id'],
                'product_name'     : combo['product_name'],
                'pincode'          : combo['pincode'],
                'area_name'        : combo['area_name'],
                'latitude'         : combo['latitude'],
                'longitude'        : combo['longitude'],
                'order_dow'        : int(row['order_dow']),
                'order_hour_of_day': int(row['order_hour_of_day']),
                'demand'           : float(row['demand']),
            })

ts_df = pd.DataFrame(all_rows)
ts_df = ts_df.sort_values(
    ['product_id','pincode','datetime']
).reset_index(drop=True)

print(f"Time series shape : {ts_df.shape}")
print(f"Date range        : {ts_df['datetime'].min()} → {ts_df['datetime'].max()}")
print(ts_df[['datetime','product_id','pincode','demand']].head(8))

Time series shape : (2059538, 11)
Date range        : 2024-01-01 00:00:00 → 2024-06-30 23:00:00
             datetime  product_id  pincode  demand
0 2024-01-01 00:00:00        4605   380006    2.00
1 2024-01-01 01:00:00        4605   380006    2.00
2 2024-01-01 02:00:00        4605   380006    2.00
3 2024-01-01 04:00:00        4605   380006    2.00
4 2024-01-01 05:00:00        4605   380006    1.00
5 2024-01-01 06:00:00        4605   380006    2.00
6 2024-01-01 07:00:00        4605   380006    4.00
7 2024-01-01 08:00:00        4605   380006    4.00


In [12]:
# add after your calendar features cell
AREA_DENSITY = {
    380009: 0.18,   # Navrangpura — high
    380015: 0.17,   # Satellite
    380051: 0.15,   # Bodakdev
    380006: 0.14,   # Paldi
    380013: 0.12,   # Gota
    380019: 0.10,   # Vastrapur
    380021: 0.08,   # Maninagar
    382210: 0.03,   # Bakrol
    382330: 0.02,   # Naroda
    382435: 0.01,   # Bareja — low
}
ts_df['area_density'] = ts_df['pincode'].map(AREA_DENSITY)
print("Area density feature added")
print(ts_df[['pincode','area_density']].drop_duplicates())

Area density feature added
       pincode  area_density
0       380006          0.14
4342    380009          0.18
8658    380013          0.12
12948   380015          0.17
17316   380019          0.10
21554   380021          0.08
25792   380051          0.15
30108   382210          0.03
33826   382330          0.02
37648   382435          0.01


In [14]:
# ── Cell 5 — verify no leakage before adding features ─────────────────
# demand should vary week to week for same product+pincode+dow+hour

sample = ts_df[
    (ts_df['product_id'] == ts_df['product_id'].iloc[0]) &
    (ts_df['pincode']    == ts_df['pincode'].iloc[0])
].head(20)

print("Same product+pincode across weeks — demand should vary:")
print(sample[['datetime','order_dow','order_hour_of_day','demand']])
print(f"\nDemand std (should be > 0): {sample['demand'].std():.2f}")

Same product+pincode across weeks — demand should vary:
              datetime  order_dow  order_hour_of_day  demand
0  2024-01-01 00:00:00          0                  0    2.00
1  2024-01-01 01:00:00          0                  1    2.00
2  2024-01-01 02:00:00          0                  2    2.00
3  2024-01-01 04:00:00          0                  4    2.00
4  2024-01-01 05:00:00          0                  5    1.00
5  2024-01-01 06:00:00          0                  6    2.00
6  2024-01-01 07:00:00          0                  7    4.00
7  2024-01-01 08:00:00          0                  8    4.00
8  2024-01-01 09:00:00          0                  9    5.00
9  2024-01-01 10:00:00          0                 10    5.00
10 2024-01-01 11:00:00          0                 11    5.00
11 2024-01-01 12:00:00          0                 12    5.00
12 2024-01-01 13:00:00          0                 13    5.00
13 2024-01-01 14:00:00          0                 14    5.00
14 2024-01-01 15:00:00       

In [15]:
sample_full = ts_df[
    (ts_df['product_id'] == ts_df['product_id'].iloc[0]) &
    (ts_df['pincode']    == ts_df['pincode'].iloc[0])
]

print(f"Total rows     : {len(sample_full)}")
print(f"Weeks covered  : {sample_full['week_number'].nunique()}")
print(f"Min demand     : {sample_full['demand'].min():.2f}")
print(f"Max demand     : {sample_full['demand'].max():.2f}")
print(f"Avg demand     : {sample_full['demand'].mean():.2f}")
print(f"Std deviation  : {sample_full['demand'].std():.2f}")

Total rows     : 4342
Weeks covered  : 26
Min demand     : 0.00
Max demand     : 7.00
Avg demand     : 3.34
Std deviation  : 1.54


In [16]:
demand_df.duplicated(subset=['product_id','pincode','order_dow','order_hour_of_day']).sum()  

0

In [17]:
# ── Cell 6 — cyclical time features ───────────────────────────────────
ts_df['hour_sin']  = np.sin(2 * np.pi * ts_df['order_hour_of_day'] / 24)
ts_df['hour_cos']  = np.cos(2 * np.pi * ts_df['order_hour_of_day'] / 24)
ts_df['dow_sin']   = np.sin(2 * np.pi * ts_df['order_dow'] / 7)
ts_df['dow_cos']   = np.cos(2 * np.pi * ts_df['order_dow'] / 7)
ts_df['week_sin']  = np.sin(2 * np.pi * ts_df['week_number'] / 52)
ts_df['week_cos']  = np.cos(2 * np.pi * ts_df['week_number'] / 52)
ts_df['month']     = ts_df['datetime'].dt.month

print("Cyclical features added")
print(ts_df[['order_hour_of_day','hour_sin','hour_cos']].head(5))

Cyclical features added
   order_hour_of_day  hour_sin  hour_cos
0                  0      0.00      1.00
1                  1      0.26      0.97
2                  2      0.50      0.87
3                  4      0.87      0.50
4                  5      0.97      0.26


In [18]:
# ── Cell 7 — calendar features ────────────────────────────────────────
ts_df['is_weekend']   = ts_df['order_dow'].isin([5, 6]).astype(int)
ts_df['is_morning']   = ts_df['order_hour_of_day'].between(6,  11).astype(int)
ts_df['is_afternoon'] = ts_df['order_hour_of_day'].between(12, 17).astype(int)
ts_df['is_evening']   = ts_df['order_hour_of_day'].between(18, 22).astype(int)
ts_df['is_night']     = ts_df['order_hour_of_day'].isin(
                            [23,0,1,2,3,4,5]).astype(int)

festival_dates = [
    '2024-01-14','2024-01-26','2024-03-25',
    '2024-04-14','2024-08-15','2024-10-02',
    '2024-11-01','2024-11-15'
]
ts_df['date_str']    = ts_df['datetime'].dt.strftime('%Y-%m-%d')
ts_df['is_festival'] = ts_df['date_str'].isin(festival_dates).astype(int)

print("Calendar features added")
print(f"Weekend rows  : {ts_df['is_weekend'].sum():,}")
print(f"Festival rows : {ts_df['is_festival'].sum():,}")

Calendar features added
Weekend rows  : 588,900
Festival rows : 45,263


In [19]:
# ── Cell 8 — lag features (leak-free) ────────────────────────────────
grp = ts_df.groupby(['product_id','pincode'])['demand']

ts_df['lag_1']   = grp.shift(1)    # 1 hour ago
ts_df['lag_24']  = grp.shift(24)   # same hour yesterday
ts_df['lag_48']  = grp.shift(48)   # same hour 2 days ago
ts_df['lag_168'] = grp.shift(168)  # same hour last week

# LEAKAGE CHECK — must be close to 0%
exact = (ts_df['lag_168'] == ts_df['demand']).mean()
print(f"Leakage check — lag_168 == demand: {exact*100:.2f}%")
print("Must be near 0% — if so we are good!\n")

print(ts_df[['datetime','demand','lag_1','lag_24','lag_168']].head(10))

Leakage check — lag_168 == demand: 29.03%
Must be near 0% — if so we are good!

             datetime  demand  lag_1  lag_24  lag_168
0 2024-01-01 00:00:00    2.00    NaN     NaN      NaN
1 2024-01-01 01:00:00    2.00   2.00     NaN      NaN
2 2024-01-01 02:00:00    2.00   2.00     NaN      NaN
3 2024-01-01 04:00:00    2.00   2.00     NaN      NaN
4 2024-01-01 05:00:00    1.00   2.00     NaN      NaN
5 2024-01-01 06:00:00    2.00   1.00     NaN      NaN
6 2024-01-01 07:00:00    4.00   2.00     NaN      NaN
7 2024-01-01 08:00:00    4.00   4.00     NaN      NaN
8 2024-01-01 09:00:00    5.00   4.00     NaN      NaN
9 2024-01-01 10:00:00    5.00   5.00     NaN      NaN


In [20]:
# ── Cell 9 — rolling features (leak-free) ────────────────────────────
# shift(1) inside rolling ensures current row never leaks into window

grp = ts_df.groupby(['product_id','pincode'])['demand']

ts_df['rolling_mean_24']  = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=1).mean())
ts_df['rolling_mean_168'] = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=1).mean())
ts_df['rolling_mean_720'] = grp.transform(
    lambda x: x.shift(1).rolling(720, min_periods=1).mean())
ts_df['rolling_std_24']   = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=2).std().fillna(0))
ts_df['rolling_std_168']  = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=2).std().fillna(0))
ts_df['rolling_max_24']   = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=1).max())
ts_df['rolling_max_168']  = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=1).max())
ts_df['ewma_24']  = grp.transform(
    lambda x: x.shift(1).ewm(span=24,  adjust=False).mean())
ts_df['ewma_168'] = grp.transform(
    lambda x: x.shift(1).ewm(span=168, adjust=False).mean())

print("Rolling features added")
print(ts_df[['datetime','demand',
             'rolling_mean_24','rolling_mean_168','ewma_24']].head(8))

Rolling features added
             datetime  demand  rolling_mean_24  rolling_mean_168  ewma_24
0 2024-01-01 00:00:00    2.00              NaN               NaN      NaN
1 2024-01-01 01:00:00    2.00             2.00              2.00     2.00
2 2024-01-01 02:00:00    2.00             2.00              2.00     2.00
3 2024-01-01 04:00:00    2.00             2.00              2.00     2.00
4 2024-01-01 05:00:00    1.00             2.00              2.00     2.00
5 2024-01-01 06:00:00    2.00             1.80              1.80     1.92
6 2024-01-01 07:00:00    4.00             1.83              1.83     1.93
7 2024-01-01 08:00:00    4.00             2.14              2.14     2.09


In [23]:
# ── Cell 10 — fetch weather  ────────────────────────
def fetch_weather(lat, lon, start='2024-01-01', end='2024-06-30'):
    url    = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude"  : lat,
        "longitude" : lon,
        "start_date": start,
        "end_date"  : end,
        "hourly"    : "temperature_2m,precipitation,relativehumidity_2m",
        "timezone"  : "Asia/Kolkata"
    }
    data = requests.get(url, params=params).json()
    return pd.DataFrame({
        'datetime'   : pd.to_datetime(data['hourly']['time']),
        'temperature': data['hourly']['temperature_2m'],
        'rainfall'   : data['hourly']['precipitation'],
        'humidity'   : data['hourly']['relativehumidity_2m'],
    })

print("Fetching Ahmedabad weather 2024...")
weather_df = fetch_weather(23.0225, 72.5714)
weather_df['is_raining'] = (weather_df['rainfall'] > 0.5).astype(int)
weather_df['is_hot']     = (weather_df['temperature'] > 35).astype(int)

print(f"Weather shape: {weather_df.shape}")
print(weather_df.head(5))

weather_df.to_csv(r'F:\projectss\personal\blinkit_project\data\external/ahmedabad_weather_2024.csv', index=False)
print("Weather saved!")

Fetching Ahmedabad weather 2024...
Weather shape: (4368, 6)
             datetime  temperature  rainfall  humidity  is_raining  is_hot
0 2024-01-01 00:00:00        17.50      0.00        80           0       0
1 2024-01-01 01:00:00        16.80      0.00        84           0       0
2 2024-01-01 02:00:00        16.00      0.00        88           0       0
3 2024-01-01 03:00:00        15.60      0.00        88           0       0
4 2024-01-01 04:00:00        15.50      0.00        88           0       0
Weather saved!


In [24]:
weather_df.duplicated(subset=['datetime']).sum()

0

In [25]:
# ── Cell 11 — merge weather ───────────────────────────────────────────
ts_df['datetime']      = pd.to_datetime(ts_df['datetime'])
weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])

ts_df = ts_df.merge(
    weather_df[['datetime','temperature','rainfall',
                'humidity','is_raining','is_hot']],
    on='datetime', how='left'
)

# fill missing weather with sensible defaults
ts_df['temperature'] = ts_df['temperature'].fillna(ts_df['temperature'].median())
ts_df['rainfall']    = ts_df['rainfall'].fillna(0)
ts_df['humidity']    = ts_df['humidity'].fillna(ts_df['humidity'].median())
ts_df['is_raining']  = ts_df['is_raining'].fillna(0)
ts_df['is_hot']      = ts_df['is_hot'].fillna(0)

print("Weather merged")
print(f"Missing temp values: {ts_df['temperature'].isnull().sum()}")
print(ts_df[['datetime','temperature','rainfall','is_raining']].head(5))

Weather merged
Missing temp values: 0
             datetime  temperature  rainfall  is_raining
0 2024-01-01 00:00:00        17.50      0.00           0
1 2024-01-01 01:00:00        16.80      0.00           0
2 2024-01-01 02:00:00        16.00      0.00           0
3 2024-01-01 04:00:00        15.50      0.00           0
4 2024-01-01 05:00:00        15.40      0.00           0


In [26]:
# ── Cell 12 — drop nulls from lag features ────────────────────────────
print(f"Shape before dropna: {ts_df.shape}")

ts_df = ts_df.dropna(subset=[
    'lag_1','lag_24','lag_168',
    'rolling_mean_24','rolling_mean_168'
])

print(f"Shape after dropna : {ts_df.shape}")
print(f"Null count total   : {ts_df.isnull().sum().sum()}")

Shape before dropna: (2059538, 44)
Shape after dropna : (1975538, 44)
Null count total   : 0


In [94]:
# ── Cell 13 — final feature list and save ─────────────────────────────
FEATURE_COLS = [
    'hour_sin','hour_cos','dow_sin','dow_cos',
    'week_sin','week_cos','month',
    'is_weekend','is_morning','is_afternoon',
    'is_evening','is_night','is_festival',
    'lag_1','lag_24','lag_48','lag_168',
    'rolling_mean_24','rolling_mean_168','rolling_mean_720',
    'rolling_std_24','rolling_std_168',
    'rolling_max_24','rolling_max_168',
    'ewma_24','ewma_168',
    'temperature','rainfall','humidity',
    'is_raining','is_hot','area_density',
]

print(f"Total features : {len(FEATURE_COLS)}")
print(f"X shape        : {ts_df[FEATURE_COLS].shape}")
print(f"y shape        : {ts_df['demand'].shape}")
print(f"\nDemand stats:\n{ts_df['demand'].describe()}")

ts_df.to_csv(r'F:\projectss\personal\blinkit_project\data\processed\features_df_fixed.csv', index=False)



Total features : 32
X shape        : (1975538, 32)
y shape        : (1975538,)

Demand stats:
count   1975538.00
mean          3.27
std           1.48
min           1.00
25%           2.00
50%           3.00
75%           4.00
max           8.00
Name: demand, dtype: float64


In [95]:
import pandas as pd
ts_df = pd.read_csv(r'F:\projectss\personal\blinkit_project\data\processed\features_df_fixed.csv')
print("Shape:", ts_df.shape)
print("Leakage check:", (ts_df['lag_168'] == ts_df['demand']).mean() * 100, "%")
print("Demand stats:\n", ts_df['demand'].describe())

Shape: (1975538, 44)
Leakage check: 37.96970749233879 %
Demand stats:
 count   1975538.00
mean          3.27
std           1.48
min           1.00
25%           2.00
50%           3.00
75%           4.00
max           8.00
Name: demand, dtype: float64


In [106]:
ts_df

,datetime,week_number,product_id,product_name,pincode,area_name,latitude,longitude,order_dow,order_hour_of_day,demand,area_density,hour_sin,hour_cos,dow_sin,dow_cos,week_sin,week_cos,month,is_weekend,is_morning,is_afternoon,is_evening,is_night,date_str,is_festival,lag_1,lag_24,lag_48,lag_168,rolling_mean_24,rolling_mean_168,rolling_mean_720,rolling_std_24,rolling_std_168,rolling_max_24,rolling_max_168,ewma_24,ewma_168,temperature,rainfall,humidity,is_raining,is_hot
0,2024-01-08 01:00:00,2,4605,Yellow Onions,380006,Paldi,23.02,72.57,0,1,3.00,0.14,0.26,0.97,0.00,1.00,0.24,0.97,1,0,0,0,0,1,2024-01-08,0,3.00,1.00,1.00,2.00,3.50,3.32,3.32,1.53,1.26,5.00,5.00,3.74,3.14,18.50,0.00,78,0,0
1,2024-01-08 02:00:00,2,4605,Yellow Onions,380006,Paldi,23.02,72.57,0,2,2.00,0.14,0.50,0.87,0.00,1.00,0.24,0.97,1,0,0,0,0,1,2024-01-08,0,3.00,1.00,1.00,2.00,3.58,3.33,3.32,1.44,1.26,5.00,5.00,3.68,3.14,18.00,0.00,82,0,0
2,2024-01-08 04:00:00,2,4605,Yellow Onions,380006,Paldi,23.02,72.57,0,4,2.00,0.14,0.87,0.50,0.00,1.00,0.24,0.97,1,0,0,0,0,1,2024-01-08,0,2.00,1.00,1.00,2.00,3.62,3.33,3.31,1.38,1.26,5.00,5.00,3.54,3.13,17.30,0.00,87,0,0
3,2024-01-08 05:00:00,2,4605,Yellow Onions,380006,Paldi,23.02,72.57,0,5,2.00,0.14,0.97,0.26,0.00,1.00,0.24,0.97,1,0,0,0,0,1,2024-01-08,0,2.00,1.00,1.00,2.00,3.67,3.33,3.30,1.31,1.26,5.00,5.00,3.42,3.11,17.00,0.00,88,0,0
4,2024-01-08 06:00:00,2,4605,Yellow Onions,380006,Paldi,23.02,72.57,0,6,3.00,0.14,1.00,0.00,0.00,1.00,0.24,0.97,1,0,1,0,0,0,2024-01-08,0,2.00,1.00,1.00,2.00,3.71,3.33,3.30,1.23,1.26,5.00,5.00,3.31,3.10,18.00,0.00,84,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975533,2024-06-30 19:00:00,26,49683,Cucumber Kirby,382435,Bareja,22.92,72.65,6,19,1.00,0.01,-0.97,0.26,-0.78,0.62,-0.00,-1.00,6,1,0,0,1,0,2024-06-30,0,3.00,2.00,2.00,2.00,2.12,2.06,1.78,0.80,0.82,3.00,4.00,2.33,2.02,27.30,0.30,94,0,0
1975534,2024-06-30 20:00:00,26,49683,Cucumber Kirby,382435,Bareja,22.92,72.65,6,20,2.00,0.01,-0.87,0.50,-0.78,0.62,-0.00,-1.00,6,1,0,0,1,0,2024-06-30,0,1.00,3.00,3.00,3.00,2.08,2.05,1.78,0.83,0.82,3.00,4.00,2.23,2.01,27.10,0.10,94,0,0
1975535,2024-06-30 21:00:00,26,49683,Cucumber Kirby,382435,Bareja,22.92,72.65,6,21,1.00,0.01,-0.71,0.71,-0.78,0.62,-0.00,-1.00,6,1,0,0,1,0,2024-06-30,0,2.00,2.00,3.00,2.00,2.04,2.05,1.78,0.81,0.82,3.00,4.00,2.21,2.01,27.00,0.00,94,0,0
1975536,2024-06-30 22:00:00,26,49683,Cucumber Kirby,382435,Bareja,22.92,72.65,6,22,2.00,0.01,-0.50,0.87,-0.78,0.62,-0.00,-1.00,6,1,0,0,1,0,2024-06-30,0,1.00,2.00,3.00,2.00,2.00,2.04,1.78,0.83,0.82,3.00,4.00,2.11,1.99,26.90,0.10,92,0,0


In [ ]:
'''# convert to datetime
ts_df['datetime'] = pd.to_datetime(ts_df['datetime'])

result = ts_df[
    (ts_df['area_name'] == 'Navrangpura') &
    (ts_df['order_dow'] == 6)   # Sunday
].copy()

# extract only date
result['date'] = result['datetime'].dt.date

# total Sunday demand per day
sunday_orders = (
    result.groupby('date')['demand']
    .sum()
    .reset_index()
    .sort_values('date')
)

print(sunday_orders)'''

          date  demand
0   2024-01-14 5605.00
1   2024-01-21 5145.00
2   2024-01-28 4895.00
3   2024-02-04 4034.00
4   2024-02-11 4034.00
5   2024-02-18 3890.00
6   2024-02-25 5423.00
7   2024-03-03 4898.00
8   2024-03-10 5089.00
9   2024-03-17 3841.00
10  2024-03-24 5634.00
11  2024-03-31 5353.00
12  2024-04-07 4122.00
13  2024-04-14 4064.00
14  2024-04-21 4064.00
15  2024-04-28 4348.00
16  2024-05-05 4815.00
17  2024-05-12 4647.00
18  2024-05-19 4323.00
19  2024-05-26 4924.00
20  2024-06-02 4025.00
21  2024-06-09 4323.00
22  2024-06-16 4505.00
23  2024-06-23 4687.00
24  2024-06-30 5259.00


In [98]:
import pandas as pd
import numpy as np

np.random.seed(42)

ts_df = pd.read_csv(
    r'F:\projectss\personal\blinkit_project\data\processed\features_df_fixed.csv'
)

selected = pd.read_csv(
    r'F:\projectss\personal\blinkit_project\data\external\selected_ahmedabad_pincodes.csv'
)

# average hourly demand
avg = (
    ts_df.groupby(['product_id', 'product_name', 'pincode'])['demand']
    .mean()
    .reset_index()
)

# convert hourly demand → approximate daily demand
avg['avg_daily_real'] = avg['demand'] * 24

# realistic stock coverage:
# 0.5 to 3 days inventory
stock_multiplier = np.random.choice(
    [0.5, 1.0, 1.5, 2.0, 3.0],
    size=len(avg),
    p=[0.10, 0.30, 0.35, 0.20, 0.05]
)

# generate stock
avg['current_stock'] = (
    avg['avg_daily_real'] * stock_multiplier
    + np.random.normal(0, 3, len(avg))
)

# clean unrealistic values
avg['current_stock'] = (
    avg['current_stock']
    .clip(lower=2)
    .round(0)
    .astype(int)
)

# optional cap to avoid unrealistic huge stock
avg['current_stock'] = avg['current_stock'].clip(upper=250)

# area names
pincode_name = dict(zip(selected['pincode'], selected['area_name']))
avg['area_name'] = avg['pincode'].map(pincode_name)

# final dataframe
stock_df = avg[
    ['product_id', 'product_name',
     'pincode', 'area_name',
     'current_stock']
].copy()

print(f"Generated stock for {len(stock_df)} product-area combinations")
print(stock_df.sample(10).to_string(index=False))

# save
stock_df.to_csv(
    r'F:\projectss\personal\blinkit_project\data\processed\stock_levels.csv',
    index=False
)

print("\nSaved stock_levels.csv")

Generated stock for 500 product-area combinations
 product_id                    product_name  pincode   area_name  current_stock
      13176          Bag of Organic Bananas   380019   Vastrapur            113
      44359      Organic Small Bunch Celery   382435      Bareja             23
      30489                 Original Hummus   380006       Paldi             38
      22935            Organic Yellow Onion   380009 Navrangpura             47
      28985           Michigan Organic Kale   380021   Maninagar            106
      27966             Organic Raspberries   380019   Vastrapur             89
      35951 Organic Unsweetened Almond Milk   382435      Bareja             41
       5876                   Organic Lemon   380021   Maninagar            158
      45007                Organic Zucchini   382435      Bareja             41
      13176          Bag of Organic Bananas   380006       Paldi            244

Saved stock_levels.csv
